# Notebook 05: Automated Model Risk Audit Toolkit
## XGBoost Training, SHAP Explainability & Quantitative Audit Diagnostics

**Capstone Stage 1 — Core Contribution | Modules 9, 10, 12**  
**Dataset:** Polish Companies Bankruptcy (UCI ID 365)  
**Author:** Srini | Imperial College London — Professional Certificate in ML & AI

---

## Overview

This notebook implements the **Automated Model Risk Audit Toolkit** — the novel capstone contribution. It applies quantitative explainability diagnostics to an XGBoost credit early warning system, producing structured audit findings against calibrated governance thresholds.

The four diagnostic modules are:

| # | Diagnostic | Metric | Regulatory Alignment |
|---|-----------|--------|---------------------|
| A | Explanation Stability | Bootstrap variance of SHAP values | NIST AI RMF GOVERN 1.2 |
| B | Explanation Drift | Wasserstein / KL / KS distance | PRA SS1/23 §3.4 |
| C | Explanation Sensitivity (ESR) | Explanation Sensitivity Ratio | SR 11-7 §IV |
| D | Local Consistency | Behavioural clustering of SHAP vectors | EU AI Act Art. 13 |

**Novel contribution:** The Explanation Sensitivity Ratio (ESR) — measuring whether explanations are more sensitive to perturbation than predictions, a brittleness signal invisible to prediction-level diagnostics alone.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from sklearn.cluster import KMeans
from scipy.stats import ks_2samp, wasserstein_distance
from scipy.special import rel_entr
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
COLOURS = {'bankrupt': '#d62728', 'solvent': '#1f77b4'}
RAG = {'red': '#d62728', 'amber': '#FF8F00', 'green': '#2E7D32'}
np.random.seed(42)

# Governance thresholds (from proposal)
THRESHOLDS = {
    'stability_amber': 0.15, 'stability_red': 0.30,
    'drift_amber': 0.10,     'drift_red': 0.25,
    'esr_amber': 1.5,        'esr_red': 3.0,
    'stress_amber': 0.80,    'stress_red': 0.60
}

def rag_colour(value, amber, red, invert=False):
    """Return RAG colour for a metric value."""
    if not invert:
        if value >= red:   return RAG['red']
        if value >= amber: return RAG['amber']
        return RAG['green']
    else:  # lower is worse (e.g. stress rank correlation)
        if value <= red:   return RAG['red']
        if value <= amber: return RAG['amber']
        return RAG['green']

print('Libraries loaded. Governance thresholds set.')
print('Thresholds:', THRESHOLDS)

---
## 1. XGBoost Model Training

In [ ]:
# Load data
df = pd.read_csv('../data/polish_bankruptcy.csv')
with open('../data/feature_map.json') as f:
    feature_map = json.load(f)
feature_cols = [c for c in df.columns if c.startswith('X')]
X_raw = df[feature_cols].values
y = df['target'].values

# Temporal proxy split: first 60% = early period (t1), last 40% = later period (t2)
# (In real deployment this would be actual annual cohorts)
split_idx = int(len(df) * 0.6)
X_t1, y_t1 = X_raw[:split_idx], y[:split_idx]
X_t2, y_t2 = X_raw[split_idx:], y[split_idx:]

# Train/test split within t1
X_train, X_test, y_train, y_test = train_test_split(X_t1, y_t1, test_size=0.25, random_state=42, stratify=y_t1)

preprocessor = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())])
X_train_pp = preprocessor.fit_transform(X_train)
X_test_pp  = preprocessor.transform(X_test)
X_t2_pp    = preprocessor.transform(X_t2)  # For drift analysis

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_pp, y_train)

# XGBoost — primary model under audit
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc', random_state=42,
    use_label_encoder=False, verbosity=0
)
xgb_model.fit(X_train_sm, y_train_sm)

test_proba = xgb_model.predict_proba(X_test_pp)[:,1]
test_auc   = roc_auc_score(y_test, test_proba)
test_f1    = f1_score(y_test, (test_proba>=0.5).astype(int))

print(f'XGBoost Model Performance (Test Set):')
print(f'  AUC-ROC : {test_auc:.4f}')
print(f'  F1 (bankrupt): {test_f1:.4f}')
print(classification_report(y_test, (test_proba>=0.5).astype(int), target_names=['Solvent','Bankrupt']))

---
## 2. SHAP Explainability Engine

In [ ]:
# Compute SHAP values — use TreeExplainer for XGBoost (exact, fast)
explainer = shap.TreeExplainer(xgb_model)

# Sample for SHAP computation (full test set)
shap_values_test = explainer.shap_values(X_test_pp)
shap_values_t2   = explainer.shap_values(X_t2_pp[:500])  # Subset for drift analysis

print(f'SHAP values computed: test={shap_values_test.shape}, t2_sample={shap_values_t2.shape}')

# Feature names with credit analyst labels
feat_labels = [feature_map.get(f'X{i+1}', f'X{i+1}').replace('_', ' ') for i in range(len(feature_cols))]

# Global SHAP summary
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Mean absolute SHAP values
mean_abs_shap = np.abs(shap_values_test).mean(axis=0)
top_idx = np.argsort(mean_abs_shap)[::-1][:15]
top_feat_labels = [feat_labels[i][:40] for i in top_idx]

axes[0].barh(range(15), mean_abs_shap[top_idx][::-1], color='#2196F3', alpha=0.85)
axes[0].set_yticks(range(15))
axes[0].set_yticklabels([f[:35] for f in top_feat_labels[::-1]], fontsize=8)
axes[0].set_xlabel('Mean |SHAP value|')
axes[0].set_title('Global SHAP Feature Importance\n(Mean Absolute SHAP, Credit Analyst Labels)', fontweight='bold')

# SHAP beeswarm-style (manual implementation)
top5_idx = top_idx[:5]
for j, feat_idx in enumerate(top5_idx):
    shap_vals = shap_values_test[:, feat_idx]
    feat_vals = X_test_pp[:, feat_idx]
    # Colour by feature value (high = red, low = blue)
    sc = axes[1].scatter(shap_vals, [j]*len(shap_vals),
                         c=feat_vals, cmap='RdBu_r', alpha=0.4, s=8,
                         vmin=np.percentile(feat_vals, 5), vmax=np.percentile(feat_vals, 95))

axes[1].set_yticks(range(5))
axes[1].set_yticklabels([feat_labels[i][:40] for i in top5_idx], fontsize=8)
axes[1].set_xlabel('SHAP value (impact on bankruptcy probability)')
axes[1].set_title('SHAP Beeswarm — Top 5 Features\n(Red = high feature value, Blue = low)', fontweight='bold')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.colorbar(sc, ax=axes[1], label='Feature value (normalised)', shrink=0.6)

plt.suptitle('XGBoost SHAP Explainability — Polish Bankruptcy Credit EWS', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/05_shap_global.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Diagnostic A: Explanation Stability — Bootstrap Variance

**Regulatory alignment:** NIST AI RMF GOVERN 1.2 — Risk identification, measurement, and documentation  
**Finding criterion:** Normalised bootstrap variance > 0.15 (Amber), > 0.30 (Red)

In [ ]:
B = 50  # Bootstrap iterations (increase to 200+ for production)
np.random.seed(42)

shap_bootstrap = np.zeros((B, X_test_pp.shape[1]))
for b in range(B):
    idx_b = np.random.choice(len(X_test_pp), size=len(X_test_pp), replace=True)
    shap_b = explainer.shap_values(X_test_pp[idx_b])
    shap_bootstrap[b] = np.abs(shap_b).mean(axis=0)

# Normalised variance: var / mean^2 — makes it scale-independent
mean_shap = shap_bootstrap.mean(axis=0)
var_shap  = shap_bootstrap.var(axis=0)
norm_var  = np.where(mean_shap > 1e-8, var_shap / (mean_shap**2 + 1e-8), 0)

# Top 15 by mean SHAP importance
top15 = np.argsort(mean_shap)[::-1][:15]
stability_colours = [rag_colour(norm_var[i], THRESHOLDS['stability_amber'], THRESHOLDS['stability_red'])
                     for i in top15]

ci_low  = np.percentile(shap_bootstrap, 2.5, axis=0)
ci_high = np.percentile(shap_bootstrap, 97.5, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Normalised variance RAG
labels_top15 = [feat_labels[i][:35] for i in top15]
bars = axes[0].barh(range(15), norm_var[top15][::-1], color=stability_colours[::-1], alpha=0.85, edgecolor='black', linewidth=0.5)
axes[0].set_yticks(range(15))
axes[0].set_yticklabels(labels_top15[::-1], fontsize=8)
axes[0].axvline(THRESHOLDS['stability_amber'], color=RAG['amber'], linestyle='--', linewidth=1.5, label='Amber threshold')
axes[0].axvline(THRESHOLDS['stability_red'],   color=RAG['red'],   linestyle='--', linewidth=1.5, label='Red threshold')
axes[0].set_xlabel('Normalised Bootstrap Variance (Var/Mean²)')
axes[0].set_title('Diagnostic A: Explanation Stability\nRAG Findings (NIST AI RMF GOVERN 1.2)', fontweight='bold')
axes[0].legend(fontsize=8)

# Bootstrap distribution for top 3 features
for j, feat_idx in enumerate(top15[:3]):
    axes[1].errorbar(j, mean_shap[feat_idx],
                     yerr=[[mean_shap[feat_idx]-ci_low[feat_idx]],
                           [ci_high[feat_idx]-mean_shap[feat_idx]]],
                     fmt='o', color=stability_colours[j], capsize=8, markersize=10, linewidth=2.5)
axes[1].set_xticks(range(3))
axes[1].set_xticklabels([feat_labels[i][:25] for i in top15[:3]], fontsize=8)
axes[1].set_ylabel('Mean |SHAP| with Bootstrap 95% CI')
axes[1].set_title('Bootstrap 95% CI — Top 3 Features\n(Wider CI = less stable explanation)', fontweight='bold')

plt.suptitle('Diagnostic A: SHAP Explanation Stability — Bootstrap Variance Analysis',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/05_diagnostic_A_stability.png', dpi=150, bbox_inches='tight')
plt.show()

n_red = (norm_var[top15] >= THRESHOLDS['stability_red']).sum()
n_amber = ((norm_var[top15] >= THRESHOLDS['stability_amber']) & (norm_var[top15] < THRESHOLDS['stability_red'])).sum()
print(f'\nDiagnostic A — Stability Findings:')
print(f'  RED findings (variance > {THRESHOLDS["stability_red"]}): {n_red} features')
print(f'  AMBER findings (variance > {THRESHOLDS["stability_amber"]}): {n_amber} features')
print(f'  GREEN (stable): {15 - n_red - n_amber} features')

---
## Diagnostic B: Explanation Drift — Wasserstein / KL / KS

**Regulatory alignment:** PRA SS1/23 §3.4 — Ongoing monitoring of model behaviour  
**Finding criterion:** Wasserstein distance > 0.10 (Amber), > 0.25 (Red)

In [ ]:
# Drift: compare SHAP distributions between t1 (test) and t2 (later period)
drift_results = []
for i in range(len(feature_cols)):
    s_t1 = shap_values_test[:, i]
    s_t2 = shap_values_t2[:, i]

    w_dist = wasserstein_distance(s_t1, s_t2)
    ks_stat, ks_pval = ks_2samp(s_t1, s_t2)

    # KL divergence via histogram approximation
    bins = np.linspace(min(s_t1.min(), s_t2.min()), max(s_t1.max(), s_t2.max()), 50)
    h1, _ = np.histogram(s_t1, bins=bins, density=True)
    h2, _ = np.histogram(s_t2, bins=bins, density=True)
    h1 = h1 + 1e-10; h2 = h2 + 1e-10
    h1 /= h1.sum(); h2 /= h2.sum()
    kl_div = float(np.sum(rel_entr(h1, h2)))

    drift_results.append({
        'feature': f'X{i+1}',
        'label': feat_labels[i][:40],
        'wasserstein': w_dist,
        'ks_stat': ks_stat,
        'ks_pval': ks_pval,
        'kl_div': kl_div
    })

drift_df = pd.DataFrame(drift_results).sort_values('wasserstein', ascending=False)

# Visualise top drifting features
top_drift = drift_df.head(15)
drift_colours = [rag_colour(v, THRESHOLDS['drift_amber'], THRESHOLDS['drift_red'])
                 for v in top_drift['wasserstein']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(range(15), top_drift['wasserstein'].values[::-1],
             color=drift_colours[::-1], alpha=0.85, edgecolor='black', linewidth=0.5)
axes[0].set_yticks(range(15))
axes[0].set_yticklabels(top_drift['label'].values[::-1], fontsize=8)
axes[0].axvline(THRESHOLDS['drift_amber'], color=RAG['amber'], linestyle='--', linewidth=1.5, label='Amber threshold')
axes[0].axvline(THRESHOLDS['drift_red'],   color=RAG['red'],   linestyle='--', linewidth=1.5, label='Red threshold')
axes[0].set_xlabel('Wasserstein Distance (SHAP distribution, t1→t2)')
axes[0].set_title('Diagnostic B: Explanation Drift\nWasserstein Distance (PRA SS1/23 §3.4)', fontweight='bold')
axes[0].legend(fontsize=8)

# Three-metric comparison for top 8
top8 = drift_df.head(8)
x_pos = np.arange(len(top8))
width = 0.3
w_norm = top8['wasserstein'] / top8['wasserstein'].max()
ks_norm = top8['ks_stat'] / top8['ks_stat'].max()
kl_norm = top8['kl_div'] / top8['kl_div'].max()

axes[1].bar(x_pos - width, w_norm,  width, label='Wasserstein (norm)', color='#2196F3', alpha=0.8)
axes[1].bar(x_pos,         ks_norm, width, label='KS statistic (norm)', color='#FF9800', alpha=0.8)
axes[1].bar(x_pos + width, kl_norm, width, label='KL divergence (norm)', color='#9C27B0', alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'X{r["feature"].replace("X","")}' for _, r in top8.iterrows()], fontsize=9)
axes[1].set_ylabel('Normalised drift score')
axes[1].set_title('Three-Metric Drift Comparison — Top 8 Features\n(Wasserstein + KL + KS triangulation)', fontweight='bold')
axes[1].legend(fontsize=8)

plt.suptitle('Diagnostic B: SHAP Explanation Drift — t1 vs t2 Temporal Windows',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/05_diagnostic_B_drift.png', dpi=150, bbox_inches='tight')
plt.show()

n_red_drift = (drift_df['wasserstein'] >= THRESHOLDS['drift_red']).sum()
n_amb_drift = ((drift_df['wasserstein'] >= THRESHOLDS['drift_amber']) & (drift_df['wasserstein'] < THRESHOLDS['drift_red'])).sum()
print(f'\nDiagnostic B — Drift Findings:')
print(f'  RED findings (Wasserstein > {THRESHOLDS["drift_red"]}): {n_red_drift} features')
print(f'  AMBER findings: {n_amb_drift} features')
print(drift_df[['feature','label','wasserstein','ks_stat','kl_div']].head(8).to_string(index=False))

---
## Diagnostic C: Explanation Sensitivity Ratio (ESR) — Novel Contribution

**Regulatory alignment:** SR 11-7 §IV — Independent challenge of model outputs  
**Definition:** ESR_i = |ΔSHAP_i(ε)| / (|ΔPrediction_i(ε)| + δ)  
**Interpretation:** ESR >> 1 means explanations are more sensitive than predictions — explanation brittleness  
**Finding criterion:** ESR > 1.5 (Amber), > 3.0 (Red)

In [ ]:
epsilon = 0.1   # Perturbation magnitude (10% of feature scale)
delta   = 1e-6  # Stability constant to prevent division by zero

# Compute ESR on a representative sample
n_esr = min(200, len(X_test_pp))  # Sample size for ESR computation
X_esr = X_test_pp[:n_esr]
shap_base = explainer.shap_values(X_esr)
pred_base = xgb_model.predict_proba(X_esr)[:,1]

esr_results = []
for i in range(len(feature_cols)):
    # Perturb feature i by epsilon
    X_perturbed = X_esr.copy()
    X_perturbed[:, i] += epsilon

    shap_pert = explainer.shap_values(X_perturbed)
    pred_pert = xgb_model.predict_proba(X_perturbed)[:,1]

    # Per-sample sensitivity
    delta_shap = np.abs(shap_pert[:, i] - shap_base[:, i])
    delta_pred = np.abs(pred_pert - pred_base)

    esr_per_sample = delta_shap / (delta_pred + delta)
    esr_mean = esr_per_sample.mean()
    pred_sens_mean = delta_pred.mean() / (epsilon + delta)

    esr_results.append({
        'feature': f'X{i+1}',
        'label': feat_labels[i][:40],
        'esr_mean': esr_mean,
        'esr_p95': np.percentile(esr_per_sample, 95),
        'pred_sensitivity': pred_sens_mean,
        'shap_sensitivity': delta_shap.mean() / epsilon
    })

esr_df = pd.DataFrame(esr_results).sort_values('esr_mean', ascending=False)

# Top 15 ESR features
top_esr = esr_df.head(15)
esr_colours = [rag_colour(v, THRESHOLDS['esr_amber'], THRESHOLDS['esr_red'])
               for v in top_esr['esr_mean']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ESR RAG bar chart
axes[0].barh(range(15), top_esr['esr_mean'].values[::-1],
             color=esr_colours[::-1], alpha=0.85, edgecolor='black', linewidth=0.5)
axes[0].set_yticks(range(15))
axes[0].set_yticklabels(top_esr['label'].values[::-1], fontsize=8)
axes[0].axvline(1.0,                          color='grey',        linestyle=':',  linewidth=1.5, label='ESR=1 (baseline)')
axes[0].axvline(THRESHOLDS['esr_amber'],       color=RAG['amber'],  linestyle='--', linewidth=1.5, label=f'Amber (>{THRESHOLDS["esr_amber"]})')
axes[0].axvline(THRESHOLDS['esr_red'],         color=RAG['red'],    linestyle='--', linewidth=1.5, label=f'Red (>{THRESHOLDS["esr_red"]})')
axes[0].set_xlabel('Explanation Sensitivity Ratio (ESR)')
axes[0].set_title('Diagnostic C: ESR — Novel Contribution\n(SR 11-7 §IV Independent Challenge)', fontweight='bold')
axes[0].legend(fontsize=8)

# Prediction vs Explanation sensitivity scatter
sc = axes[1].scatter(esr_df['pred_sensitivity'], esr_df['shap_sensitivity'],
                     c=esr_df['esr_mean'], cmap='RdYlGn_r',
                     s=80, alpha=0.8, edgecolors='black', linewidth=0.5,
                     vmin=0, vmax=THRESHOLDS['esr_red'])
# ESR=1 line
max_val = max(esr_df['pred_sensitivity'].max(), esr_df['shap_sensitivity'].max())
axes[1].plot([0, max_val], [0, max_val], 'k--', alpha=0.4, label='ESR=1 (equal sensitivity)')

# Annotate high-ESR features
for _, row in esr_df[esr_df['esr_mean'] > THRESHOLDS['esr_amber']].head(5).iterrows():
    axes[1].annotate(row['feature'], (row['pred_sensitivity'], row['shap_sensitivity']),
                     fontsize=7, xytext=(3,3), textcoords='offset points')

axes[1].set_xlabel('Prediction Sensitivity')
axes[1].set_ylabel('Explanation (SHAP) Sensitivity')
axes[1].set_title('Prediction vs Explanation Sensitivity\n(Points above diagonal: ESR > 1)', fontweight='bold')
axes[1].legend(fontsize=8)
plt.colorbar(sc, ax=axes[1], label='ESR', shrink=0.8)

plt.suptitle('Diagnostic C: Explanation Sensitivity Ratio (ESR) — Independent Challenge Evidence',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/05_diagnostic_C_ESR.png', dpi=150, bbox_inches='tight')
plt.show()

n_red_esr = (esr_df['esr_mean'] >= THRESHOLDS['esr_red']).sum()
n_amb_esr = ((esr_df['esr_mean'] >= THRESHOLDS['esr_amber']) & (esr_df['esr_mean'] < THRESHOLDS['esr_red'])).sum()
print(f'\nDiagnostic C — ESR Findings:')
print(f'  RED findings (ESR > {THRESHOLDS["esr_red"]}): {n_red_esr} features')
print(f'  AMBER findings (ESR > {THRESHOLDS["esr_amber"]}): {n_amb_esr} features')
print(f'\nTop 10 highest ESR features (most brittle explanations):')
print(esr_df[['feature','label','esr_mean','esr_p95','pred_sensitivity']].head(10).to_string(index=False))

---
## Diagnostic D: Local Explanation Consistency — Behavioural Clustering

**Regulatory alignment:** EU AI Act Art. 13 — Transparency of automated individual decisions  
**Method:** K-means clustering on SHAP vectors to identify subpopulations with different explanation profiles

In [ ]:
from sklearn.decomposition import PCA

# Elbow method to select k
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(shap_values_test)
    inertias.append(km.inertia_)

# Optimal k by elbow
k_opt = 4  # From elbow inspection
km_final = KMeans(n_clusters=k_opt, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(shap_values_test)

# PCA for 2D visualisation
pca = PCA(n_components=2)
shap_2d = pca.fit_transform(shap_values_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Elbow plot
axes[0].plot(K_range, inertias, 'o-', color='#2196F3', linewidth=2, markersize=8)
axes[0].axvline(k_opt, color='red', linestyle='--', alpha=0.7, label=f'Selected k={k_opt}')
axes[0].set_xlabel('Number of clusters k'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method — Optimal k Selection', fontweight='bold')
axes[0].legend()

# 2D cluster scatter
cluster_palette = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0','#00BCD4','#795548']
for c in range(k_opt):
    mask = cluster_labels == c
    axes[1].scatter(shap_2d[mask, 0], shap_2d[mask, 1],
                    color=cluster_palette[c], alpha=0.5, s=15,
                    label=f'Cluster {c} (n={mask.sum()})')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
axes[1].set_title('SHAP Behavioural Clusters (2D PCA)\n(EU AI Act Art. 13 — Local explanation consistency)', fontweight='bold')
axes[1].legend(fontsize=8)

# Cluster profiles — mean SHAP for top features
top5 = np.argsort(np.abs(shap_values_test).mean(axis=0))[::-1][:5]
cluster_means = np.zeros((k_opt, 5))
for c in range(k_opt):
    cluster_means[c] = shap_values_test[cluster_labels==c][:, top5].mean(axis=0)

im = axes[2].imshow(cluster_means, cmap='RdBu_r', aspect='auto',
                    vmin=-np.abs(cluster_means).max(), vmax=np.abs(cluster_means).max())
axes[2].set_xticks(range(5))
axes[2].set_xticklabels([feat_labels[i][:20] for i in top5], rotation=30, ha='right', fontsize=7)
axes[2].set_yticks(range(k_opt))
axes[2].set_yticklabels([f'Cluster {c}' for c in range(k_opt)])
axes[2].set_title('Cluster SHAP Profiles\n(Red=↑ bankruptcy risk, Blue=↓)', fontweight='bold')
plt.colorbar(im, ax=axes[2], label='Mean SHAP value', shrink=0.8)

plt.suptitle('Diagnostic D: Local Explanation Consistency — Behavioural Clustering',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/05_diagnostic_D_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

# Cluster bankruptcy rates
print('Cluster Analysis — Bankruptcy Rates and SHAP Profiles:')
for c in range(k_opt):
    mask = cluster_labels == c
    bkr_rate = y_test[mask].mean()
    print(f'  Cluster {c}: n={mask.sum():4d}, bankruptcy rate={bkr_rate:.1%}')

---
## Audit Findings Scorecard — RAG Summary

In [ ]:
# Consolidated RAG scorecard
findings = [
    {'Diagnostic': 'A: Explanation Stability',
     'Metric': 'Normalised Bootstrap Variance',
     'Value': f'{n_red} RED / {n_amber} AMBER features',
     'Threshold': f'Amber>{THRESHOLDS["stability_amber"]} Red>{THRESHOLDS["stability_red"]}',
     'Status': 'RED' if n_red > 0 else ('AMBER' if n_amber > 0 else 'GREEN'),
     'Framework': 'NIST AI RMF GOVERN 1.2'},
    {'Diagnostic': 'B: Explanation Drift',
     'Metric': 'Wasserstein Distance (t1→t2)',
     'Value': f'{n_red_drift} RED / {n_amb_drift} AMBER features',
     'Threshold': f'Amber>{THRESHOLDS["drift_amber"]} Red>{THRESHOLDS["drift_red"]}',
     'Status': 'RED' if n_red_drift > 0 else ('AMBER' if n_amb_drift > 0 else 'GREEN'),
     'Framework': 'PRA SS1/23 §3.4'},
    {'Diagnostic': 'C: Explanation Sensitivity (ESR)',
     'Metric': 'Explanation Sensitivity Ratio',
     'Value': f'{n_red_esr} RED / {n_amb_esr} AMBER features',
     'Threshold': f'Amber>{THRESHOLDS["esr_amber"]} Red>{THRESHOLDS["esr_red"]}',
     'Status': 'RED' if n_red_esr > 0 else ('AMBER' if n_amb_esr > 0 else 'GREEN'),
     'Framework': 'SR 11-7 §IV'},
    {'Diagnostic': 'D: Local Consistency',
     'Metric': 'Behavioural Cluster Heterogeneity',
     'Value': f'{k_opt} distinct explanation clusters',
     'Threshold': '>2 clusters triggers review',
     'Status': 'AMBER' if k_opt > 2 else 'GREEN',
     'Framework': 'EU AI Act Art. 13'},
]

scorecard_df = pd.DataFrame(findings)

# Visual scorecard
fig, ax = plt.subplots(figsize=(16, 4))
ax.axis('off')
col_widths = [2.5, 2.5, 2.5, 2.5, 1.0, 2.0]
table = ax.table(
    cellText=scorecard_df.values,
    colLabels=scorecard_df.columns,
    cellLoc='center', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 2.5)

# Colour status cells
status_col_idx = list(scorecard_df.columns).index('Status')
for row_idx, status in enumerate(scorecard_df['Status']):
    colour = '#FFCDD2' if status == 'RED' else ('#FFF9C4' if status == 'AMBER' else '#C8E6C9')
    table[row_idx+1, status_col_idx].set_facecolor(colour)
    table[row_idx+1, status_col_idx].set_text_props(fontweight='bold')

# Header row
for col_idx in range(len(scorecard_df.columns)):
    table[0, col_idx].set_facecolor('#1565C0')
    table[0, col_idx].set_text_props(color='white', fontweight='bold')

ax.set_title('Automated Model Risk Audit — Findings Scorecard\nPolish Bankruptcy EWS | XGBoost | SHAP Explainability Diagnostics',
             fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../reports/05_audit_scorecard.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAUDIT FINDINGS SCORECARD')
print('=' * 100)
print(scorecard_df.to_string(index=False))

---
## Summary

This notebook has implemented the complete **Automated Model Risk Audit Toolkit** — the novel capstone contribution — demonstrating:

1. XGBoost as the primary model under audit, with strong AUC performance on the Polish Bankruptcy dataset
2. SHAP explainability engine producing global and local attributions in credit analyst language
3. Four quantitative diagnostics — Stability, Drift, ESR, Clustering — each aligned to a named regulatory expectation
4. A consolidated RAG findings scorecard separating audit findings from analytics outputs

The **Explanation Sensitivity Ratio (ESR)** is the methodological novelty: it reveals explanation brittleness invisible to prediction-level diagnostics alone, directly supporting independent challenge under SR 11-7.

---
## Next Notebook

→ **Notebook 06:** Stage 2 — Bayesian Optimisation & BBO Challenge